In [3]:
# =============================================================================
# 🐾 PAWTY AI BACKEND - GOOGLE COLAB PRO EDITION
# =============================================================================

import os
import sys
import subprocess
import threading
import base64
from io import BytesIO
from PIL import Image
import torch
import asyncio

# --- 1. 极速安装依赖 (只在第一次运行时安装) ---
print("🚀 正在初始化环境...")
try:
    import diffusers
    import fastapi
    import pyngrok
    print("✅ 依赖库已安装")
except ImportError:
    print("⬇️ 正在安装依赖库 (diffusers, fastapi, uvicorn, pyngrok)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "diffusers", "transformers", "accelerate", "safetensors",
                           "fastapi", "uvicorn", "pyngrok", "python-multipart", "nest_asyncio"])
    print("✅ 安装完成")

import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from diffusers import StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler

# --- 2. 加载 AI 模型引擎 (Pro GPU版) ---
class PetStyleService:
    def __init__(self, model_id="Lykon/dreamshaper-8"):
        print(f"🔄 正在加载模型: {model_id}...")

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🖥️  运行设备: {self.device} (应该是 cuda)")

        if self.device == "cpu":
            print("⚠️ 警告: 未检测到 GPU！生成速度会非常慢。请检查运行时设置。")

        # 使用 float16 精度 (Colab Pro A100/T4 上最快)
        self.pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            use_safetensors=True
        )

        # 使用 DPM++ 调度器 (生成质量更好，步数更少)
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipe.scheduler.config,
            use_karras_sigmas=True,
            algorithm_type="dpmsolver++"
        )

        self.pipe = self.pipe.to(self.device)
        self.pipe.enable_attention_slicing() # 显存优化
        print("✨ 模型加载完毕，准备就绪！")

    def generate(self, init_image, style_prompt, species, strength=0.75):
        # 预处理图片：缩放到 512x512 以保证速度和显存安全
        init_image = init_image.convert("RGB").resize((512, 512))

        # 你的“严格风格”提示词逻辑
        full_prompt = (
            f"({style_prompt}:1.4), masterpiece, best quality, 8k, "
            f"(cute {species}), (animal only), detailed fur, cinematic lighting"
        )
        negative_prompt = "human, person, man, woman, hands, feet, text, watermark, bad anatomy, blur, lowres"

        with torch.autocast("cuda"):
            result = self.pipe(
                prompt=full_prompt,
                image=init_image,
                strength=strength,      # 0.75 意味着重绘 75%
                guidance_scale=9.0,     # 强迫模型听从提示词
                negative_prompt=negative_prompt,
                num_inference_steps=25  # DPM++ 25步足够了
            ).images[0]

        return result

# --- 3. 配置 FastAPI 服务器 ---
app = FastAPI()

# 允许跨域 (解决本地网页访问问题)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 风格映射表 (对应 HTML 里的 ID)
STYLE_MAP = {
    "1": {"p": "Oil Painting, thick strokes, textured canvas", "s": 0.75},
    "2": {"p": "Pixar style, disney 3d render, cute, vibrant", "s": 0.75},
    "3": {"p": "Cyberpunk, neon lights, mechanical parts, sci-fi", "s": 0.80},
    "4": {"p": "Pencil sketch, graphite, monochrome, rough lines", "s": 0.65},
    "5": {"p": "Ghibli style, anime, vibrant colors, detailed background", "s": 0.75}
}

service = PetStyleService() # 初始化模型

@app.post("/stylize")
async def process_image(
    style_id: str = Form(...),
    species: str = Form(...),
    image: UploadFile = File(...)
):
    print(f"📩 收到请求: {species} | 风格: {style_id}")

    # 读取上传的图片
    img_bytes = await image.read()
    input_pil = Image.open(BytesIO(img_bytes))

    # 获取风格配置
    style = STYLE_MAP.get(style_id, STYLE_MAP["1"])

    # 生成图片
    output_pil = service.generate(input_pil, style["p"], species, style["s"])

    # 转 Base64
    buffered = BytesIO()
    output_pil.save(buffered, format="PNG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

    return {"status": "success", "image_base64": img_b64}

# --- 4. 启动服务器 (修复版 for L4/Colab) ---
import uvicorn
import nest_asyncio

# ================= 在此处填入你的 TOKEN =================
NGROK_TOKEN = "36lRCtcHBQ8soISUzEnvrC9uYMz_4MHNArj68JaNrP8zDNdaU"
# ======================================================

# 1. 设置 Ngrok
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill() # 关闭旧隧道

# 2. 创建隧道 (端口 8011)
tunnel = ngrok.connect(8011)
public_url = tunnel.public_url

print(f"\n✅ =============================================")
print(f"🎉 服务器已启动 (L4 GPU 加速中)！")
print(f"👉 请复制这个 URL 到 index.html: {public_url}")
print(f"=============================================\n")

# 3. 解决 Colab 报错的关键修复：
nest_asyncio.apply()

# 使用 Config 和 Server 对象，配合 await 启动
config = uvicorn.Config(app, port=8011, host="0.0.0.0")
server = uvicorn.Server(config)
await server.serve()  # <--- 注意这里用了 await，而不是 .run()

/usr/local/lib/python3.12/dist-packages/IPython/core/compilerop.py:101: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)


🚀 正在初始化环境...
✅ 依赖库已安装
🔄 正在加载模型: Lykon/dreamshaper-8...
🖥️  运行设备: cuda (应该是 cuda)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.


✨ 模型加载完毕，准备就绪！

✅ =============================================
🎉 服务器已启动 (L4 GPU 加速中)！
👉 请复制这个 URL 到 index.html: https://dwight-unexpectant-aloofly.ngrok-free.dev



...

  0%|          | 0/20 [00:00<?, ?it/s]

INFO:     207.38.249.232:0 - "POST /stylize HTTP/1.1" 200 OK
📩 收到请求: cat | 风格: 5


  0%|          | 0/18 [00:00<?, ?it/s]

INFO:     207.38.249.232:0 - "POST /stylize HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1182]
